In [1]:
# =========================
# MDS → クラスタリング 一括実行（OH/FP 自動処理・new_MDS保存版）
# - 相関→距離(1 - cor)→cmdscale（古典MDS）
# - NbClust（ward.D2, Euclidean）× 指標6種（silhouette, dunn, gap, ch, db, ptbiserial）
# - 次元条件: top3 / cumeig（正の固有値寄与率で累積 ≥ 80%）
# - 出力: new_MDS_YYYYMMDD/<dataset>/ に CSV と 図を保存
# =========================

# --- 必要パッケージ ---
if (!require(NbClust)) { install.packages("NbClust"); library(NbClust) }
if (!require(ggplot2)) { install.packages("ggplot2"); library(ggplot2) }
if (!require(ggrepel)) { install.packages("ggrepel"); library(ggrepel) }
if (!require(scales))  { install.packages("scales");  library(scales) }
if (!require(MASS))    { install.packages("MASS");    library(MASS) }     # isoMDS 用（必要なら）
if (!require(mclust))  { install.packages("mclust");  library(mclust) }   # ARI

# --- 設定（入力パス）---
# 作業ディレクトリにファイルがあれば "" のままでもOK。固定パスを使う場合は base_dir を指定してください。
base_dir  <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
dataset_files <- c("preprocessed_features_OH.csv", "preprocessed_features_FP.csv")

# --- 共通設定 ---
set.seed(42)  # 再現性
index_list   <- c("silhouette", "dunn", "gap", "ch", "db", "ptbiserial")  # 指標6種
ts_day       <- format(Sys.Date(), "%Y%m%d")                               # 日付タグ（フォルダ用）
root_out     <- file.path(base_dir, paste0("STEP3_new_MDS_", ts_day))            # 例: .../new_MDS_20250904
dir.create(root_out, showWarnings = FALSE, recursive = TRUE)

# --- ヘルパー ---
pct <- function(x) sprintf("%.2f%%", 100 * x)

read_numeric_matrix <- function(path){
  df <- read.delim(path, header = TRUE, sep = ",", row.names = 1,
                   as.is = TRUE, strip.white = FALSE)
  is_num <- vapply(df, function(x) !is.character(x), logical(1))
  if (!any(is_num)) stop("数値列が見つかりません: ", path)
  df[, is_num, drop = FALSE]
}

compute_mds <- function(numData, k_max = 300){
  NS <- nrow(numData); NF <- ncol(numData)
  message(sprintf("[Info] rows=%d, cols=%d", NS, NF))
  nonNApct <- sum(!is.na(as.matrix(numData))) / (NS*NF) * 100
  message(sprintf("[Info] Non-NA percentage: %.2f%%", nonNApct))

  corData <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
  corData[is.na(corData)] <- 0
  ddata <- dist(1 - corData)

  mds_k <- min(k_max, max(1, NF - 1))
  mdsdata <- cmdscale(ddata, k = mds_k, eig = TRUE)

  # 固有値（全体）と、正の固有値のみでの寄与率（プロットやcumeig用）
  eig_all <- mdsdata$eig
  S_all   <- sum(eig_all)
  share_all <- eig_all / ifelse(S_all == 0, 1, S_all)

  eig_pos <- eig_all[eig_all > 0]
  S_pos   <- sum(eig_pos)
  share_pos_full <- rep(NA_real_, length(eig_all))
  if (length(eig_pos) > 0 && S_pos > 0) {
    share_pos_full[eig_all > 0] <- eig_all[eig_all > 0] / S_pos
  }

  cum_all <- cumsum(share_all)
  tmp <- share_pos_full; tmp[is.na(tmp)] <- 0
  cum_pos <- cumsum(tmp)

  list(mdsdata = mdsdata,
       eig_all = eig_all,
       share_all = share_all,
       share_pos_full = share_pos_full,
       cum_all = cum_all,
       cum_pos = cum_pos)
}

choose_dims <- function(share_pos_full, cum_pos, target = 0.80){
  # 正の固有値寄与で top3 / cumeig の軸数を決める
  pos_idx <- which(!is.na(share_pos_full) & share_pos_full > 0)
  top3 <- min(3, max(1, length(pos_idx)))

  if (length(pos_idx) == 0) {
    return(list(top3 = 1, cumeig = 3))  # フォールバック
  }
  # cumeig: 累積 ≥ target に初めて達する位置
  k_cum <- which(cum_pos[pos_idx] >= target)[1]
  if (is.na(k_cum)) k_cum <- length(pos_idx)
  cumeig <- max(1, k_cum)

  list(top3 = top3, cumeig = cumeig)
}

save_eigen_csv_and_plots <- function(out_dir, suffix_tag, ts_tag, eig_all){
  # 正の固有値でスクリープロット
  eig_pos <- eig_all[eig_all > 0]
  if (length(eig_pos) > 0) {
    df_eig <- data.frame(
      Dim  = seq_along(eig_pos),
      Eig  = as.numeric(eig_pos)
    )
    df_eig$Prop <- df_eig$Eig / sum(df_eig$Eig)
    df_eig$Cum  <- cumsum(df_eig$Prop)

    # 1) スクリープロット（棒＋累積線）
    p_scree <- ggplot(df_eig, aes(x = Dim, y = Prop)) +
      geom_col(width = 0.85, alpha = 0.9) +
      geom_line(aes(y = Cum), linewidth = 1.1) +
      geom_point(aes(y = Cum), size = 2) +
      scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                         sec.axis = sec_axis(~ ., name = "Cumulative share")) +
      labs(title = "Scree plot (positive eigenvalues only)",
           subtitle = paste0("Classical MDS  |  ", suffix_tag),
           x = "Dimension", y = "Variance share") +
      theme_minimal(base_size = 12) +
      theme(panel.grid.minor = element_blank(),
            axis.title = element_text(face = "bold"),
            plot.title = element_text(face = "bold"))
    ggsave(file.path(out_dir, paste0("MDS_scree_", suffix_tag, "_", ts_tag, ".png")),
           p_scree, width = 7, height = 5, dpi = 300)
    ggsave(file.path(out_dir, paste0("MDS_scree_", suffix_tag, "_", ts_tag, ".pdf")),
           p_scree, width = 7, height = 5, device = cairo_pdf)

    # 2) 固有値トップ50
    topN <- min(50, nrow(df_eig))
    df_eig50 <- df_eig[1:topN, , drop = FALSE]
    p_bar50 <- ggplot(df_eig50, aes(x = Dim, y = Prop)) +
      geom_col(width = 0.85, alpha = 0.95) +
      labs(title = paste0("Top-", topN, " eigenvalue shares"),
           subtitle = paste0("Classical MDS  |  ", suffix_tag),
           x = "Dimension", y = "Variance share") +
      scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
      theme_minimal(base_size = 12) +
      theme(panel.grid.minor = element_blank(),
            axis.title = element_text(face = "bold"),
            plot.title = element_text(face = "bold"))
    ggsave(file.path(out_dir, paste0("MDS_bar_top", topN, "_", suffix_tag, "_", ts_tag, ".png")),
           p_bar50, width = 7, height = 5, dpi = 300)
    ggsave(file.path(out_dir, paste0("MDS_bar_top", topN, "_", suffix_tag, "_", ts_tag, ".pdf")),
           p_bar50, width = 7, height = 5, device = cairo_pdf)
  } else {
    warning("正の固有値が無いため、スクリープロットとTop50図はスキップします。")
  }
}

run_nbclust_one_index <- function(mds_points, cindex, out_dir, suffix_tag, ts_tag){
  clustEst <- tryCatch({
    NbClust(
      data = mds_points, diss = NULL, distance = "euclidean",
      min.nc = 2, max.nc = 100, method = "ward.D2", index = cindex
    )
  }, error = function(e) { warning(paste("Index", cindex, "failed:", e$message)); NULL })
  if (is.null(clustEst)) return(NULL)

  best_nc <- clustEst$Best.nc[1]
  grpname <- as.factor(clustEst$Best.partition)

  # 割当CSV
  fn_assign <- paste0(cindex, "_DataGrp_", suffix_tag, ".csv")
  write.csv(grpname, file = file.path(out_dir, fn_assign))

  # 1-2 軸散布図（楕円付き）
  df_plot <- data.frame(
    MDS1    = mds_points[, 1],
    MDS2    = mds_points[, 2],
    Cluster = grpname,
    ID      = rownames(mds_points)
  )
  rng <- range(c(df_plot$MDS1, df_plot$MDS2), na.rm = TRUE)
  pad <- diff(rng) * 0.05; lims <- c(min(rng) - pad, max(rng) + pad)

  p_scatter <- ggplot(df_plot, aes(x = MDS1, y = MDS2, color = Cluster)) +
    stat_ellipse(aes(group = Cluster), type = "norm", level = 0.95,
                 linewidth = 0.6, alpha = 0.25) +
    geom_point(size = 2.2, alpha = 0.8) +
    coord_equal(xlim = lims, ylim = lims, expand = TRUE) +
    scale_color_hue(h = c(0, 360), l = 55, c = 90) +
    labs(title = "MDS (Dim1 vs Dim2) — Ward.D2 × NbClust",
         subtitle = paste0("Index: ", cindex, "  |  Best k = ", best_nc, "  |  ", suffix_tag),
         x = "MDS Dimension 1", y = "MDS Dimension 2", color = "Cluster") +
    theme_minimal(base_size = 12) +
    theme(panel.grid.major = element_line(linewidth = 0.3, linetype = "dashed"),
          panel.grid.minor = element_blank(),
          axis.title = element_text(face = "bold"),
          plot.title = element_text(face = "bold", size = 14, margin = margin(b = 4)),
          plot.subtitle = element_text(size = 11, colour = "grey30"),
          legend.position = "right",
          legend.title = element_text(face = "bold"))
  ggsave(file.path(out_dir, paste0("MDS12_scatter_", cindex, "_", suffix_tag, "_", ts_tag, ".png")),
         p_scatter, width = 7, height = 6, dpi = 300)
  ggsave(file.path(out_dir, paste0("MDS12_scatter_", cindex, "_", suffix_tag, "_", ts_tag, ".pdf")),
         p_scatter, width = 7, height = 6, device = cairo_pdf)

  # NbClust All.index 曲線
  if (!is.null(clustEst$All.index)) {
    idx_vals  <- as.numeric(clustEst$All.index)
    k_labels  <- names(clustEst$All.index)
    k_seq <- if (is.null(k_labels) || any(k_labels == "")) seq_along(idx_vals) else suppressWarnings(as.numeric(k_labels))
    if (any(is.na(k_seq))) k_seq <- seq_along(idx_vals)

    df_idx <- data.frame(k = k_seq, value = idx_vals)
    p_idx <- ggplot(df_idx, aes(x = k, y = value)) +
      geom_line(linewidth = 1) +
      geom_point(size = 2) +
      labs(title = paste0("NbClust All.index — ", cindex),
           subtitle = paste0("Ward.D2 / Euclidean  |  ", suffix_tag),
           x = "k", y = "Index value") +
      theme_minimal(base_size = 12) +
      theme(panel.grid.minor = element_blank(),
            axis.title = element_text(face = "bold"),
            plot.title = element_text(face = "bold"))
    ggsave(file.path(out_dir, paste0("NbClust_AllIndex_", cindex, "_", suffix_tag, "_", ts_tag, ".png")),
           p_idx, width = 7, height = 5, dpi = 300)
    ggsave(file.path(out_dir, paste0("NbClust_AllIndex_", cindex, "_", suffix_tag, "_", ts_tag, ".pdf")),
           p_idx, width = 7, height = 5, device = cairo_pdf)
  }

  # クラスタサイズ棒グラフ
  df_size <- as.data.frame(table(Cluster = grpname))
  p_size <- ggplot(df_size, aes(x = Cluster, y = Freq, fill = Cluster)) +
    geom_col(width = 0.7, alpha = 0.9, show.legend = FALSE) +
    geom_text(aes(label = Freq), vjust = -0.3, size = 3.6) +
    labs(title = "Cluster sizes",
         subtitle = paste0("Assignment by NbClust best k  |  ", suffix_tag),
         x = "Cluster", y = "Count") +
    theme_minimal(base_size = 12) +
    theme(panel.grid.minor = element_blank(),
          axis.title = element_text(face = "bold"),
          plot.title = element_text(face = "bold"))
  ggsave(file.path(out_dir, paste0("Cluster_sizes_", cindex, "_", suffix_tag, "_", ts_tag, ".png")),
         p_size, width = 6.5, height = 5, dpi = 300)
  ggsave(file.path(out_dir, paste0("Cluster_sizes_", cindex, "_", suffix_tag, "_", ts_tag, ".pdf")),
         p_size, width = 6.5, height = 5, device = cairo_pdf)

  list(best_nc = best_nc, grp = grpname, clustEst = clustEst)
}

# =========================
# メイン：OH / FP を順に実行
# =========================
for (ifname in dataset_files) {
  # データセット名（OH / FP）推定
  dataset_stem <- sub("\\.csv$", "", ifname)                  # preprocessed_features_OH
  suffix_tag <- if (grepl("OH", ifname, ignore.case = TRUE)) "OH" else
                if (grepl("FP", ifname, ignore.case = TRUE)) "FP" else dataset_stem

  # 出力フォルダ（データごと）
  out_dir <- file.path(root_out, paste0(suffix_tag))
  dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
  ts_tag <- format(Sys.time(), "%Y%m%d_%H%M%S")               # ファイル名用の時刻タグ

  message("\n==============================")
  message(sprintf("=== Processing: %s (%s) ===", ifname, suffix_tag))
  message("==============================")

  # 読み込み
  in_path <- file.path(base_dir, ifname)
  readData <- read_numeric_matrix(in_path)

  # MDS
  m <- compute_mds(readData, k_max = 300)
  mdsdata        <- m$mdsdata
  eig_all        <- m$eig_all
  share_all      <- m$share_all
  share_pos_full <- m$share_pos_full
  cum_all        <- m$cum_all
  cum_pos        <- m$cum_pos

  # 第1〜3軸の簡易レポート（全固有値基準）
  p1_all <- share_all[1]; p2_all <- share_all[2]
  cum3_all <- sum(share_all[1:min(3, length(share_all))], na.rm = TRUE)
  cat("【全固有値基準】第1:", pct(p1_all), "  第2:", pct(p2_all),
      "  1–2合計:", pct(p1_all + p2_all), "  1–3累積:", pct(cum3_all), "\n")

  # 寄与一覧 CSV（全基準／正の基準）
  df_eig_out <- data.frame(
    Dim           = seq_along(eig_all),
    Eigenvalue    = eig_all,
    Share_all     = share_all,
    Share_pos     = share_pos_full,
    CumShare_all  = cum_all,
    CumShare_pos  = cum_pos
  )
  write.csv(df_eig_out, file.path(out_dir, paste0("MDS_eigen_contributions_", suffix_tag, "_", ts_tag, ".csv")),
            row.names = FALSE)

  # 図（スクリープロット、Top50）
  save_eigen_csv_and_plots(out_dir, suffix_tag, ts_tag, eig_all)

  # ====== NbClust：指標ごと（全次元のmdsdata$pointsを使う可視化）======
  for (cindex in index_list) {
    cat("\n=== Index:", cindex, "===\n")
    res <- run_nbclust_one_index(mds_points = mdsdata$points,
                                 cindex = cindex,
                                 out_dir = out_dir,
                                 suffix_tag = suffix_tag,
                                 ts_tag = ts_tag)
    if (!is.null(res)) {
      cat("Best.nc =", res$best_nc, "\n")
      cat("Saved figures & assignment under: ", normalizePath(out_dir), "\n")
    }
  }

  # ====== top3 vs cumeig（≥0.8）での一致度（ARI） ======
  dims <- choose_dims(share_pos_full, cum_pos, target = 0.80)
  k_top3   <- dims$top3
  k_cumeig <- dims$cumeig

  # 利用可能次元の上限でカット
  avail_dims <- ncol(mdsdata$points)
  k_top3   <- min(k_top3,   avail_dims)
  k_cumeig <- min(k_cumeig, avail_dims)

  coords_list <- list(
    top3   = mdsdata$points[, 1:k_top3,   drop = FALSE],
    cumeig = mdsdata$points[, 1:k_cumeig, drop = FALSE]
  )

  # 座標CSV（条件ごと）
  for (cond in names(coords_list)) {
    write.csv(coords_list[[cond]],
              file.path(out_dir, paste0("MDScoords_", cond, "_", suffix_tag, "_", ts_tag, ".csv")))
  }

  # 条件×指標でクラスタリングして保存
  assignments <- list()
  for (cond in names(coords_list)) {
    X <- coords_list[[cond]]
    rownames(X) <- rownames(mdsdata$points)

    for (cindex in index_list) {
      clustEst <- tryCatch({
        NbClust(data = X, diss = NULL, distance = "euclidean",
                min.nc = 2, max.nc = 100, method = "ward.D2", index = cindex)
      }, error = function(e) { warning(paste("failed:", cond, cindex, e$message)); NULL })
      if (is.null(clustEst)) next

      grp <- as.factor(clustEst$Best.partition)
      # 割当て保存
      fn_assign2 <- paste0("ClusterAssign_", cond, "_", cindex, "_",
                           suffix_tag, "_", ts_tag, ".csv")
      write.csv(data.frame(Variable = names(grp), Cluster = grp),
                file.path(out_dir, fn_assign2), row.names = FALSE)

      assignments[[paste(cond, cindex, sep = "_")]] <- grp
    }
  }

  # ARI: top3 と cumeig の一致度
  ari_rows <- lapply(index_list, function(ix) {
    a <- assignments[[paste0("top3_", ix)]]
    b <- assignments[[paste0("cumeig_", ix)]]
    ari <- if (!is.null(a) && !is.null(b)) mclust::adjustedRandIndex(a, b) else NA_real_
    data.frame(Index = ix, ARI_top3_vs_cumeig = ari)
  })
  df_ari <- do.call(rbind, ari_rows)
  write.csv(df_ari, file.path(out_dir, paste0("ARI_top3_vs_cumeig_", suffix_tag, "_", ts_tag, ".csv")),
            row.names = FALSE)
  print(df_ari)

  cat("\n✅ Done: ", suffix_tag, " → ", normalizePath(out_dir), "\n")
}

cat("\n🎉 全データの処理が完了しました → ", normalizePath(root_out), "\n")


Loading required package: NbClust

Loading required package: ggplot2

Loading required package: ggrepel

Loading required package: scales

Loading required package: MASS

Loading required package: mclust

Package 'mclust' version 6.1.1
Type 'citation("mclust")' for citing this R package in publications.



=== Processing: preprocessed_features_OH.csv (OH) ===


[Info] rows=1046, cols=394

[Info] Non-NA percentage: 98.69%



<U+3010><U+5168><U+56FA><U+6709><U+5024><U+57FA><U+6E96><U+3011><U+7B2C>1: 21.51%   <U+7B2C>2: 4.73%   1<U+2013>2<U+5408><U+8A08>: 26.23%   1<U+2013>3<U+7D2F><U+7A4D>: 30.52% 

=== Index: silhouette ===
Best.nc = 2 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/OH 

=== Index: dunn ===


Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 7 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 7 rows containing missing values or values outside the scale range
(`geom_path()`)."


Best.nc = 27 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/OH 

=== Index: gap ===
Best.nc = 2 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/OH 

=== Index: ch ===
Best.nc = 2 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/OH 

=== Index: db ===
Best.nc = 3 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/OH 

=== Index: ptbiserial ===


Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_path()`)."


...

Best.nc = 100 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/FP 

=== Index: dunn ===


Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 3 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 3 rows containing missing values or values outside the scale range
(`geom_path()`)."


Best.nc = 27 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/FP 

=== Index: gap ===
Best.nc = 2 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/FP 

=== Index: ch ===
Best.nc = 5 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/FP 

=== Index: db ===


Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calcula

Best.nc = 100 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/FP 

=== Index: ptbiserial ===
Best.nc = 6 
Saved figures & assignment under:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/FP 
       Index ARI_top3_vs_cumeig
1 silhouette          0.7269810
2       dunn          0.6659305
3        gap          1.0000000
4         ch          0.7316243
5         db          0.7316243
6 ptbiserial          0.8081430

<U+2705> Done:  FP  <U+2192>  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904/FP 

<U+0001F389> <U+5168><U+30C7><U+30FC><U+30BF><U+306E><U+51E6><U+7406><U+304C><U+5B8C><U+4E86><U+3057><U+307E><U+3057><U+305F> <U+2192>  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/new_MDS_20250904 


In [2]:
# =========================
# 独立診断スクリプト：列が減った理由の特定 & ログ出力
# - 入力: preprocessed_features_OH.csv / preprocessed_features_FP.csv
# - 出力: diagnostics_<DATASET>_<timestamp>/ にCSVレポート一式
# =========================

options(stringsAsFactors = FALSE)

# ---- 設定 ----
base_dir  <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
in_files  <- c("preprocessed_features_OH.csv", "preprocessed_features_FP.csv")
# NEWの想定NA閾値（必要なければ NULL に）
na_thr_for_check <- 0.30   # 例: 0.30。チェック不要なら NA にする

# ---- ユーティリティ ----
ts_tag <- format(Sys.time(), "%Y%m%d_%H%M%S")
dir_safe <- function(p){ dir.create(p, showWarnings=FALSE, recursive=TRUE); p }

read_raw <- function(path){
  # 文字列表現のNAもNA扱いで読む
  read.delim(path, header=TRUE, sep=",", row.names=1,
             as.is=TRUE, strip.white=TRUE,
             na.strings=c("NA","NaN","",".","null","NULL","N/A"))
}

is_num_col <- function(x) is.numeric(x) || is.integer(x)

fix_char_to_num <- function(x){
  # 文字列列をなるべく数値へ救済（%, カンマ、空白、±、指数表記の余計な空白など）
  if (!is.character(x)) return(x)
  y <- trimws(x)
  y <- gsub(",", "", y, fixed=TRUE)                 # 3,200 → 3200
  y <- gsub("%$", "", y)                            # "12.3%" → "12.3"
  y <- gsub("^\\+", "", y)                          # "+3.2" → "3.2"
  y[y %in% c("", ".", "NaN", "nan", "NULL", "null")] <- NA
  suppressWarnings(as.numeric(y))
}

preview_values <- function(v, n=5){
  y <- v
  y <- y[!is.na(y)]
  y <- unique(y)
  if (length(y) == 0) return(NA_character_)
  paste0(utils::head(y, n), collapse=" | ")
}

# ---- メイン ----
for (fname in in_files){
  dataset <- if (grepl("OH", fname, ignore.case=TRUE)) "OH" else
             if (grepl("FP", fname, ignore.case=TRUE)) "FP" else "DATA"
  in_path <- file.path(base_dir, fname)
  if (!file.exists(in_path)) { message("✖ Not found: ", in_path); next }

  out_dir <- dir_safe(file.path(base_dir, paste0("diagnostics_", dataset, "_", ts_tag)))

  cat("\n==============================\n")
  cat("Dataset:", dataset, "\nInput  :", in_path, "\n")
  cat("==============================\n")

  # 1) 生読み
  raw <- read_raw(in_path)
  n_row <- nrow(raw)
  n_col_all <- ncol(raw)
  cat(sprintf("[RAW] rows=%d, cols=%d\n", n_row, n_col_all))

  # 2) 生読み直後のcharacter列一覧
  char_cols_raw <- names(raw)[vapply(raw, is.character, logical(1))]
  numlike_cols_raw <- names(raw)[vapply(raw, is_num_col,  logical(1))]
  cat(sprintf("  numeric-like=%d, character=%d\n", length(numlike_cols_raw), length(char_cols_raw)))

  # 3) 文字列→数値 変換試行
  fixed <- as.data.frame(lapply(raw, fix_char_to_num), stringsAsFactors=FALSE)
  numlike_after_fix <- names(fixed)[vapply(fixed, is_num_col, logical(1))]
  char_after_fix    <- setdiff(names(fixed), numlike_after_fix)

  cat(sprintf("[FIX] numeric-like(after)=%d, character(after)=%d\n",
              length(numlike_after_fix), length(char_after_fix)))

  # 4) セット差分（どれが救済されたか／残ったか）
  rescued   <- intersect(char_cols_raw, numlike_after_fix)  # 元char → 数値化できた
  stubborn  <- intersect(char_cols_raw, char_after_fix)     # 元char → 依然char（非数値）
  dropped18_guess <- setdiff(numlike_cols_raw, numlike_after_fix) # まれだが数値→非数値化の検知

  # 5) 列別 NA率
  na_ratio <- colSums(is.na(fixed)) / nrow(fixed)

  # 6) ログテーブルを作成
  df_cols <- data.frame(
    Column          = names(raw),
    Was_Character   = names(raw) %in% char_cols_raw,
    Is_Numeric_After= names(raw) %in% numlike_after_fix,
    NA_Ratio        = as.numeric(na_ratio[names(raw)]),
    Example_Raw     = vapply(raw,  preview_values, character(1)),
    Example_Fixed   = vapply(fixed, preview_values, character(1)),
    stringsAsFactors = FALSE
  )

  write.csv(df_cols, file.path(out_dir, paste0("column_diagnostics_", dataset, ".csv")), row.names=FALSE)

  # 7) 代表的なリスト出力
  write.csv(data.frame(Rescued_Char_To_Numeric = rescued),
            file.path(out_dir, paste0("rescued_columns_", dataset, ".csv")), row.names=FALSE)
  write.csv(data.frame(Still_NonNumeric = stubborn),
            file.path(out_dir, paste0("stubborn_non_numeric_", dataset, ".csv")), row.names=FALSE)

  if (length(dropped18_guess) > 0){
    write.csv(data.frame(Numeric_Became_NonNumeric = dropped18_guess),
              file.path(out_dir, paste0("numeric_became_non_numeric_", dataset, ".csv")), row.names=FALSE)
  }

  # 8) 実際に「数値列のみ」に切り出した行列
  numData <- fixed[, numlike_after_fix, drop=FALSE]
  cat(sprintf("[NUMERIC MATRIX] rows=%d, cols=%d\n", nrow(numData), ncol(numData)))
  write.csv(numData, file.path(out_dir, paste0("numeric_matrix_", dataset, ".csv")))

  # 9) （任意）NEWのNA閾値チェック：どれだけ残るかの試算
  if (!is.null(na_thr_for_check) && !is.na(na_thr_for_check)){
    keep <- na_ratio <= na_thr_for_check
    kept_cols <- names(na_ratio)[keep]
    write.csv(
      data.frame(Variable = names(na_ratio),
                 NA_Ratio = as.numeric(na_ratio),
                 Keep_at_thr = keep),
      file.path(out_dir, paste0("na_ratio_log_thr", gsub("\\.","",sprintf("%.2f",na_thr_for_check)), "_", dataset, ".csv")),
      row.names = FALSE
    )
    cat(sprintf("[NA-THR=%.2f] keep %d / %d cols (on fixed matrix)\n",
                na_thr_for_check, sum(keep), length(keep)))
    write.csv(numData[, kept_cols, drop=FALSE],
              file.path(out_dir, paste0("numeric_matrix_thr", gsub("\\.","",sprintf("%.2f",na_thr_for_check)), "_", dataset, ".csv")))
  }

  # 10) まとめ（標準出力）
  cat("---- SUMMARY ----\n")
  cat("Total columns (raw):         ", n_col_all, "\n")
  cat("Character columns (raw):     ", length(char_cols_raw), "\n")
  cat("Rescued to numeric:          ", length(rescued), "\n")
  cat("Still non-numeric (stubborn):", length(stubborn), "\n")
  if (length(dropped18_guess) > 0)
    cat("Numeric→non-numeric (rare):  ", length(dropped18_guess), "\n")
  cat("Numeric matrix cols (after): ", length(numlike_after_fix), "\n")
  if (!is.null(na_thr_for_check) && !is.na(na_thr_for_check))
    cat(sprintf("Kept at NA-threshold=%.2f:    %d cols\n",
                na_thr_for_check, sum(na_ratio <= na_thr_for_check)))
  cat("Output dir: ", out_dir, "\n")
}

cat("\n✅ 完了：各データセットの診断CSVを出力しました。\n")



Dataset: OH 
Input  : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/preprocessed_features_OH.csv 
[RAW] rows=1046, cols=394
  numeric-like=394, character=0
[FIX] numeric-like(after)=394, character(after)=0
[NUMERIC MATRIX] rows=1046, cols=394
[NA-THR=0.30] keep 382 / 394 cols (on fixed matrix)
---- SUMMARY ----
Total columns (raw):          394 
Character columns (raw):      0 
Rescued to numeric:           0 
Still non-numeric (stubborn): 0 
Numeric matrix cols (after):  394 
Kept at NA-threshold=0.30:    382 cols
Output dir:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/diagnostics_OH_20251008_123244 

Dataset: FP 
Input  : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/preprocessed_features_FP.csv 
[RAW] rows=1046, cols=251
  numeric-like=251, character=0
[FIX] numeric-like(after)=251, character(after)=0
[NUMERIC MATRIX] rows=1046, cols=251
[NA-THR=0.30] keep 239 / 251 cols (on fixed matrix)
---- SUMMARY ----
Total columns (raw):          2